Load relevant packages:

In [ ]:
%matplotlib inline

%reset

import glob
import pickle
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import seaborn as sns
import textwrap

from matplotlib import cm
#from statistics import mean
from scipy.integrate import simpson


sys.path.append("/Users/sambe/Documents/Python_Projects/FDA/functions")
import fNIRSData
import auxFuncChPt


import importlib
importlib.reload(fNIRSData)

In [ ]:
#User defined params
location = 'GM'
agePoint = 5
numChrom = 2 #1=HbO, 2 = HbR, 3 = HbT
method = 'FF' #change point method: FPCA or FF
useAgeSpecific ='Y'

In [ ]:
#Other params
firstTrial = 1
lastTrial = 25
firstSamp = -20
lastSamp = 182

#Changes output filename
if numChrom == 1:
    chromName = 'HbO'
elif numChrom == 2:
    chromName = 'HbR'
else:
    chromName = 'HbT'

# Dependent on other user-defined params (above)
if useAgeSpecific == 'Y':
    # age specific (CPA) ROI channels
    tfceChannels = fNIRSData.getROI(agePoint, location)
    roiChannels = tfceChannels[chromName]
else:
    roiChannels = []
    for age in [5, 8, 12]:
        chans = fNIRSData.getROI(age, location)
        roiChannels.extend(chans["Both"])
    roiChannels = list(set(roiChannels))


# Channel mapping for plots later on
# channel mapping
mappingDict = {
    1: 15, 2: 18, 3: 12, 4: 16, 5: 19, 6: 34, 7: 37, 8: 31, 9: 35, 10: 38,
    11: 9, 12: 13, 13: 17, 14: 6, 15: 10, 16: 14, 17: 28, 18: 32, 19: 36,
    20: 25, 21: 29, 22: 33, 23: 3, 24: 7, 25: 11, 26: 22, 27: 26, 28: 30,
    29: 20, 30: 23, 31: 27, 32: 2, 33: 5, 34: 8
}

#DF containing all data for given age
nirsFrameFilename = '/Users/sambe/Documents/Python_Projects/FDA/dataRaw/homer2'+str(location)+str(agePoint)+'moDataFrame.pkl'
#DF containing group-derived change points for ROI channels
chromChangeptFrameFilename = '/Users/sambe/Documents/Python_Projects/FDA/data/grpChangePointDFs/ChPt'+location+str(agePoint)+'mo'+chromName+method+'.pkl'
#DF containing CPts combined for both chromophores
changeptFrameFilename = '/Users/sambe/Documents/Python_Projects/FDA/data/grpChangePointDFs/combinedChPt'+str(location)+str(agePoint)+'mo'+str(method)+'.pkl'
#DF containing group-derived volume calculations for ROI channels
volFrameFilename = '/Users/sambe/Documents/Python_Projects/FDA/data/grpVolRatioDFs/VolRatio'+location+str(agePoint)+'mo'+chromName+method+'.pkl'


Load dataframes:

In [ ]:
with open(nirsFrameFilename, 'rb') as f:
    dfFull = pickle.load(f)

with open(chromChangeptFrameFilename, 'rb') as f:
    dfChangePoints = pickle.load(f)   

with open(changeptFrameFilename, 'rb') as f:
    dfCombinedChangePoints = pickle.load(f) 

with open(volFrameFilename, 'rb') as f:
    dfVolumes = pickle.load(f)   

Obtain relevant metadata from dataframe for subsequent indexing through, and accessing objects in, the DF: 

In [ ]:
#find participant IDS, number of chromophores, number of channels and number of trials for subsequent indexing

partIDs = sorted(set(dfFull.loc[:,'ID']))
chroms = sorted(set(dfFull.loc[:,'Chrom']))
chans = sorted(set(dfFull.loc[:,'Channel']))
trialNums = sorted(set(dfFull.loc[:,'Trial']))


#find the dataframe column which contains the recorded time series
indexTS = dfFull.columns.get_loc("Time series")

In [ ]:
dfChangePointsROI = dfCombinedChangePoints[(dfCombinedChangePoints["Channel"].isin(roiChannels)) & (dfCombinedChangePoints['Chromophore'] == numChrom)]
dfChangePointsROI = dfCombinedChangePoints[(dfCombinedChangePoints['Chromophore'] == numChrom)]


#find ROI channels where changepoint has been detected
roiChannelsCPt = dfChangePointsROI['Channel'].unique().tolist()

#convert channels to BRIGHT numbering for plotting test
roiChannelsBRIGHT = [mappingDict[ch] if ch in mappingDict else ch for ch in roiChannelsCPt]

#scales for plots to match scaling in axis labels
scaleFactor = 10**6


# Loop over ROI channels
for numChan in roiChannelsCPt: 

    chanBRIGHT = roiChannelsBRIGHT[roiChannelsCPt.index(numChan)]
    
    # Obtain CPt info for channel
    dfChan = (dfChangePointsROI[dfChangePointsROI['Channel'] == numChan]).sort_values(by=['Change Point'])
    changeColumn = (dfChan.get('Change Point').tolist())
    changeColumn = [int(x) for x in changeColumn]
    changeColumn.append(lastTrial + 1)

    # Query data from chosen channel
    dataMatrix = fNIRSData.groupDataMatrix(dfFull,
                                           numChan,
                                           numChrom,
                                           firstTrial,
                                           lastTrial)

    # initialise variables
    iBlock = 0
    firstBlockTrial = 1
    blockMeans = np.empty([len(changeColumn), dfFull.iloc[0, indexTS].shape[0]])
    blockStds = np.empty_like(blockMeans)  # for standard deviation storage for CI
    lineStyles = ['-', '--', ':', '-.', (0, (5, 10)), (0, (1, 1)), (0, (3, 5, 1, 5))]

    # Calculate mean and standard deviation between change points
    for point in changeColumn:
        lastBlockTrial = point - 1
        blockMatrix = dataMatrix[firstBlockTrial - 1:lastBlockTrial - 1, ]
        blockMeans[iBlock] = np.mean(blockMatrix, axis=0)
        blockStds[iBlock] = np.std(blockMatrix, axis=0)  # Calculate std dev
        firstBlockTrial = point
        iBlock = iBlock + 1

    # Create dataframe for trial means and standard deviation
    trialFrame = pd.DataFrame(data=blockMeans)
    trialFrame = trialFrame.T
    trialFrame.columns = trialFrame.columns + 1
    trialFrame.index = np.arange(-2.0, 18.2, 0.1)

    # single figure for all three plots
    fig = plt.figure(figsize=(24, 8))  

    # Plot 1: 2D plot with error bars
    ax1 = fig.add_subplot(1, 3, 1)  
    time = trialFrame.index.values
    for i in range(trialFrame.shape[1]):
        if numChrom == 1:
            color = cm.plasma(0.2 + (0.7 * i / (trialFrame.shape[1] - 1)))
        else:
            color = cm.viridis(0.2 + (0.7 * i / (trialFrame.shape[1] - 1)))
        linestyle = lineStyles[i % len(lineStyles)]
        ax1.plot(time, (trialFrame.iloc[:, i])*scaleFactor, color=color, linestyle=linestyle, label=f'Mean response {i+1}')
        ax1.fill_between(time, (trialFrame.iloc[:, i] - blockStds[i])*scaleFactor, (trialFrame.iloc[:, i] + blockStds[i])*scaleFactor,
                         color=color, alpha=0.3)  # Confidence Interval (±std)

    ax1.set_xlabel('Time(s)')
    ax1.set_ylabel('Concentration change ($\mu$M)')
    title = (f'Mean haemodynamic responses between changepoints in Channel ' + str(chanBRIGHT) + ' across all '+str(agePoint)+' month participants')
    ax1.set_title('\n'.join(textwrap.wrap(title, width=30)))
    ax1.legend()

    # Plot 2: 3D surface of full data matrix
    ax2 = fig.add_subplot(1, 3, 2, projection="3d")  
    Y = np.arange(0, dataMatrix.shape[0])
    X = np.arange(0, dataMatrix.shape[1])
    X, Y = np.meshgrid(X, Y)

    # Apply scale factor directly to dataMatrix
    scaledDataMatrix = dataMatrix * scaleFactor
    
    ax2.plot_surface(Y, X, scaledDataMatrix, cmap=cm.viridis if numChrom == 2 else cm.plasma, lw=0.5, rstride=1, cstride=1)
    title = (f'Mean haemodynamic responses in channel '+str(chanBRIGHT)+' across all '+str(agePoint)+' month participants')
    ax2.set_title('\n'.join(textwrap.wrap(title, width=30)))
    ax2.set_xlabel('Trial')
    ax2.set_ylabel('Time (ds)')
    ax2.set_zlabel('Concentration change ($\mu$M)')

    # Plot 3: 3D surface between change points
    blockSurfGrid = np.empty([dataMatrix.shape[0]*1000, dataMatrix.shape[1]])
    for i in range(dataMatrix.shape[0]*1000):
        posTemp = ([j for j in range(len(changeColumn)) if changeColumn[j] >= (i/1000)+1])
        blockSurfGrid[i] = blockMeans[min(posTemp)]
    
    # Apply scale factor directly to dataMatrix
    scaledBlockSurfGrid = blockSurfGrid * scaleFactor

    ax3 = fig.add_subplot(1, 3, 3, projection="3d") 
    Y = np.arange(0, dataMatrix.shape[0], 0.001)
    X = np.arange(0, dataMatrix.shape[1])
    X, Y = np.meshgrid(X, Y)
    ax3.plot_surface(Y, X, scaledBlockSurfGrid, cmap=cm.viridis if numChrom == 2 else cm.plasma, lw=0.5, rstride=1000, cstride=1)
    title = (f'Mean haemodynamic responses between change points in channel '+str(chanBRIGHT)+' across all '+str(agePoint)+' month participants')
    ax3.set_title('\n'.join(textwrap.wrap(title, width=30)))
    ax3.set_xlabel('Trial')
    ax3.set_ylabel('Time (ds)')
    ax3.set_zlabel('Concentration change ($\mu$M)')

    # Adjust layout and display the figure
    plt.tight_layout()
    plt.show()


In [ ]:
channelsHbO = dfCombinedChangePoints[dfCombinedChangePoints["Chromophore"] == 1]["Channel"].unique().tolist()
channelsHbR = dfCombinedChangePoints[dfCombinedChangePoints["Chromophore"] == 2]["Channel"].unique().tolist()

roiChannelsTotal = dfCombinedChangePoints[dfCombinedChangePoints["Channel"].isin(channelsHbO) & dfCombinedChangePoints["Channel"].isin(channelsHbR)]["Channel"].unique().tolist()
numChroms = dfCombinedChangePoints['Chromophore'].unique().tolist()


#convert channels to BRIGHT numbering for plotting test
roiChannelsBRIGHT = [mappingDict[ch] if ch in mappingDict else ch for ch in roiChannelsTotal]

#scales for plots to match scaling in axis labels
scaleFactor = 10**6


for numChan in roiChannelsTotal:

    chanBRIGHT = roiChannelsBRIGHT[roiChannelsTotal.index(numChan)]

    # Create a figure for both chromophores
    fig = plt.figure(figsize=(20, 8))

    zMin, zMax = None, None  # Initialize z-axis limits

    for idx, numChrom in enumerate(numChroms):

        # Obtain CPt info for channel and chromophore
        dfChan = dfCombinedChangePoints[(dfCombinedChangePoints['Channel'] == numChan) & (dfCombinedChangePoints['Chromophore'] == numChrom)].sort_values(by=['Change Point'])
        changeColumn = dfChan['Change Point'].tolist()
        changeColumn = [int(x) for x in changeColumn]
        changeColumn.append(lastTrial + 1)

        # Query data from chosen channel and chromophore
        dataMatrix = fNIRSData.groupDataMatrix(dfFull,
                                               numChan,
                                               numChrom,
                                               firstTrial,
                                               lastTrial)

        # Initialise variables
        blockMeans = np.empty([len(changeColumn), dataMatrix.shape[1]])

        # Calculate mean between change points
        iBlock = 0
        firstBlockTrial = 1
        for point in changeColumn:
            lastBlockTrial = point - 1
            blockMatrix = dataMatrix[firstBlockTrial - 1:lastBlockTrial - 1, :]* scaleFactor
            blockMeans[iBlock] = np.mean(blockMatrix, axis=0)
            firstBlockTrial = point
            iBlock += 1

        # Generate block surface grid for 3D plot
        blockSurfGrid = np.empty([dataMatrix.shape[0] * 1000, dataMatrix.shape[1]])
        for i in range(dataMatrix.shape[0] * 1000):
            posTemp = [j for j in range(len(changeColumn)) if changeColumn[j] >= (i / 1000) + 1]
            blockSurfGrid[i] = blockMeans[min(posTemp)]

        # Create 3D surface plot for current chromophore
        ax = fig.add_subplot(1, 2, idx + 1, projection="3d")
        Y = np.arange(0, dataMatrix.shape[0], 0.001)
        X = np.arange(0, dataMatrix.shape[1])
        X, Y = np.meshgrid(X, Y)
        surf = ax.plot_surface(Y, X, blockSurfGrid, cmap=cm.viridis_r if numChrom == 2 else cm.plasma, lw=0.5, rstride=1000, cstride=1)
        
        # Respeat z-axis limits from the first plot
        if idx == 0:
            zMin, zMax = np.nanmin(blockSurfGrid), np.nanmax(blockSurfGrid)
        else:
            zMax2 = np.nanmax(blockSurfGrid)  # original max for plot 2
            zMin2 = zMax2 - (zMax - zMin)  # apply same range as plot 1
            ax.set_zlim(zMin2, zMax2)
        
        title = (f'Mean haemodynamic responses between change points in channel {chanBRIGHT}, chromophore {numChrom} across all {agePoint}mo participants')
        ax.set_title('\n'.join(textwrap.wrap(title, width=30)))
        ax.set_xlabel('Trial')
        ax.set_ylabel('Time (ds)')
        ax.set_zlabel('Concentration change ($\mu$M)')

    # Adjust layout and display
    plt.tight_layout()
    plt.show()



In [ ]:
dfChangePointsROI = dfCombinedChangePoints[(dfCombinedChangePoints["Channel"].isin(roiChannels))]
#dfChangePointsROI = dfCombinedChangePoints

channelsHbO = dfChangePointsROI[dfChangePointsROI["Chromophore"] == 1]["Channel"].unique().tolist()
channelsHbR = dfChangePointsROI[dfChangePointsROI["Chromophore"] == 2]["Channel"].unique().tolist()

roiChannelsTotal = dfChangePointsROI[dfChangePointsROI["Channel"].isin(channelsHbO) & dfChangePointsROI["Channel"].isin(channelsHbR)]["Channel"].unique().tolist()

numChroms = dfChangePointsROI['Chromophore'].unique().tolist()

#convert channels to BRIGHT numbering for plotting test
roiChannelsBRIGHT = [mappingDict[ch] if ch in mappingDict else ch for ch in roiChannelsTotal]

for numChan in roiChannelsTotal:

    chanBRIGHT = roiChannelsBRIGHT[roiChannelsTotal.index(numChan)]

    # Create a figure for both chromophores
    fig = plt.figure(figsize=(20, 8))

    z_min, z_max = None, None  # Initialize z-axis limits

    for idx, numChrom in enumerate(numChroms):

        # Obtain CPt info for channel and chromophore
        dfChan = dfCombinedChangePoints[(dfCombinedChangePoints['Channel'] == numChan) & (dfCombinedChangePoints['Chromophore'] == numChrom)].sort_values(by=['Change Point'])
        changeColumn = dfChan['Change Point'].tolist()
        changeColumn = [int(x) for x in changeColumn]
        changeColumn.append(lastTrial + 1)

        # Query data from chosen channel and chromophore
        dataMatrix = fNIRSData.groupDataMatrix(dfFull,
                                               numChan,
                                               numChrom,
                                               firstTrial,
                                               lastTrial)

        # Initialise variables
        blockMeans = np.empty([len(changeColumn), dataMatrix.shape[1]])

        # Calculate mean between change points
        iBlock = 0
        firstBlockTrial = 1
        for point in changeColumn:
            lastBlockTrial = point - 1
            blockMatrix = dataMatrix[firstBlockTrial - 1:lastBlockTrial - 1, :] * scaleFactor
            blockMeans[iBlock] = np.mean(blockMatrix, axis=0)
            firstBlockTrial = point
            iBlock += 1

        # Generate block surface grid for 3D plot
        blockSurfGrid = np.empty([dataMatrix.shape[0], dataMatrix.shape[1]])
        for i in range(dataMatrix.shape[0] * 1000):
            posTemp = [j for j in range(len(changeColumn)) if changeColumn[j] >= (i / 1000) + 1]
            blockSurfGrid[i] = blockMeans[min(min(posTemp), blockMeans.shape[0] - 1)]

        # Create 3D surface plot for current chromophore
        ax = fig.add_subplot(1, 2, idx + 1, projection="3d")
        Y = np.arange(0, dataMatrix.shape[0], 0.001)
        X = np.arange(0, dataMatrix.shape[1])
        X, Y = np.meshgrid(X, Y)
        surf = ax.plot_surface(Y, X, scaledBlockSurfGrid, cmap=cm.viridis_r if numChrom == 2 else cm.plasma, lw=0.5, rstride=1000, cstride=1)

        # Respeat z-axis limits from the first plot
        if idx == 0:
            z_min, z_max = np.nanmin(blockSurfGrid), np.nanmax(blockSurfGrid)
        else:
            z_max_2 = np.nanmax(blockSurfGrid)  # original max for plot 2
            z_min_2 = z_max_2 - (z_max - z_min)  # apply same range as plot 1
            ax.set_zlim(z_min_2, z_max_2)

        # Adjust the view for the second plot
        title = (f'Mean haemodynamic responses between change points in channel {chanBRIGHT}, chromophore {numChrom} across all {agePoint}mo participants')
        ax.set_title('\n'.join(textwrap.wrap(title, width=30)))
        ax.set_xlabel('Trial')
        ax.set_ylabel('Time (ds)')
        ax.set_zlabel('Concentration change ($\mu$M)')

    # Adjust layout and display
    plt.tight_layout()
    plt.show()


In [ ]:
dfChangePointsROI = dfCombinedChangePoints[(dfCombinedChangePoints["Channel"].isin(roiChannels))]
#dfChangePointsROI = dfCombinedChangePoints

channelsHbO = dfChangePointsROI[dfChangePointsROI["Chromophore"] == 1]["Channel"].unique().tolist()
channelsHbR = dfChangePointsROI[dfChangePointsROI["Chromophore"] == 2]["Channel"].unique().tolist()

roiChannelsTotal = dfChangePointsROI[dfChangePointsROI["Channel"].isin(channelsHbO) & dfChangePointsROI["Channel"].isin(channelsHbR)]["Channel"].unique().tolist()
numChroms = dfChangePointsROI['Chromophore'].unique().tolist()

#convert channels to BRIGHT numbering for plotting test
roiChannelsBRIGHT = [mappingDict[ch] if ch in mappingDict else ch for ch in roiChannelsTotal]

for numChan in roiChannelsTotal:

    chanBRIGHT = roiChannelsBRIGHT[roiChannelsTotal.index(numChan)]

    # Create a single figure for the current channel
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection="3d")  # Single 3D plot for both chromophores

    for numChrom in numChroms:

        # Obtain CPt info for channel and chromophore
        dfChan = dfCombinedChangePoints[(dfCombinedChangePoints['Channel'] == numChan) & (dfCombinedChangePoints['Chromophore'] == numChrom)].sort_values(by=['Change Point'])
        changeColumn = dfChan['Change Point'].tolist()
        changeColumn = [int(x) for x in changeColumn]
        changeColumn.append(lastTrial + 1)

        # Query data from chosen channel and chromophore
        dataMatrix = fNIRSData.groupDataMatrix(dfFull,
                                               numChan,
                                               numChrom,
                                               firstTrial,
                                               lastTrial)

        # Initialise variables
        blockMeans = np.empty([len(changeColumn), dataMatrix.shape[1]])

        # Calculate mean between change points
        iBlock = 0
        firstBlockTrial = 1
        for point in changeColumn:
            lastBlockTrial = point - 1
            blockMatrix = dataMatrix[firstBlockTrial - 1:lastBlockTrial - 1, :]
            blockMeans[iBlock] = np.mean(blockMatrix, axis=0)
            firstBlockTrial = point
            iBlock += 1

        # Generate block surface grid for 3D plot
        blockSurfGrid = np.empty([dataMatrix.shape[0] * 1000, dataMatrix.shape[1]])
        for i in range(dataMatrix.shape[0] * 1000):
            posTemp = [j for j in range(len(changeColumn)) if changeColumn[j] >= (i / 1000) + 1]
            blockSurfGrid[i] = blockMeans[min(posTemp)]

        # Create 3D surface plot for current chromophore
        Y = np.arange(0, dataMatrix.shape[0], 0.001)
        X = np.arange(0, dataMatrix.shape[1])
        X, Y = np.meshgrid(X, Y)
        surface = ax.plot_surface(Y, X, blockSurfGrid, cmap=cm.viridis if numChrom == 1 else cm.plasma, lw=0.5, rstride=1000, cstride=1, alpha=0.7)
        ax.set_xlabel('Trial')
        ax.set_ylabel('Time (ds)')
        ax.set_zlabel('Concentration change ($\mu$M)')

    # Rotate the view for a more horizontal angle
    ax.view_init(elev=10)  # Lower elevation for a more horizontal view

    title = f'Mean haemodynamic responses in channel {chanBRIGHT} for both chromophores across all {agePoint}mo participants'
    ax.set_title('\n'.join(textwrap.wrap(title, width=30)))

    # Adjust layout and display
    plt.tight_layout()
    plt.show()
